# Food Delivery Business Performance — Visualization Portfolio

Download the Food Delivery Business Performance CSV from the LMS **Study Material** tab. Run the one code cell below in Google Colab and upload the file when prompted. Each chart is followed by a data-driven interpretation.

In [ ]:
# Food Delivery Business Performance visualization portfolio — one Google Colab cell
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
if not csv_files:
    raise FileNotFoundError('Please upload the Food Delivery Business Performance CSV file.')

df = pd.read_csv(csv_files[0])
df.columns = df.columns.str.strip()
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='deep')

def find_column(candidates):
    normalized = {col.lower().replace('_', ' ').replace('-', ' ').strip(): col for col in df.columns}
    for candidate in candidates:
        if candidate in normalized:
            return normalized[candidate]
    for name, original in normalized.items():
        if any(candidate in name for candidate in candidates):
            return original
    return None

date_col = find_column(['order date', 'date', 'order datetime', 'order time'])
orders_col = find_column(['orders', 'order count', 'number of orders'])
revenue_col = find_column(['revenue', 'sales', 'total revenue', 'order value'])
city_col = find_column(['city', 'location'])
cuisine_col = find_column(['cuisine', 'cuisine type'])
marketing_col = find_column(['marketing spend', 'marketing', 'advertising spend'])
delivery_col = find_column(['delivery time', 'delivery time minutes', 'delivery'])
rating_col = find_column(['customer rating', 'rating', 'ratings'])
weather_col = find_column(['weather', 'weather condition'])
channel_col = find_column(['order channel', 'channel', 'order type'])

# Clean expected numeric values, allowing inputs such as '₹2,500' or '35 min'.
numeric_candidates = [col for col in [orders_col, revenue_col, marketing_col, delivery_col, rating_col] if col]
for column in numeric_candidates:
    df[column] = pd.to_numeric(df[column].astype(str).str.replace(r'[^0-9.-]', '', regex=True), errors='coerce')
if date_col:
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

print('=' * 90)
print(f'FOOD DELIVERY BUSINESS PERFORMANCE — VISUALIZATION PORTFOLIO: {csv_files[0]}')
print('=' * 90)
print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print('Columns:', df.columns.tolist())
display(df.head())

findings = []

# 1. LINE PLOT — revenue/orders trend over time.
if date_col and (revenue_col or orders_col):
    trend_value = revenue_col if revenue_col else orders_col
    trend = df.dropna(subset=[date_col]).groupby(pd.Grouper(key=date_col, freq='D'))[trend_value].sum().reset_index()
    plt.figure(figsize=(12, 5))
    plt.plot(trend[date_col], trend[trend_value], marker='o', linewidth=2, color='#1f77b4', label=f'Daily {trend_value}')
    plt.title(f'Daily {trend_value} Trend')
    plt.xlabel('Date')
    plt.ylabel(trend_value)
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    peak = trend.loc[trend[trend_value].idxmax()]
    interpretation = f'Line plot: {trend_value} peaked on {peak[date_col].date()} at {peak[trend_value]:,.2f}, showing the strongest recorded day.'
    print(interpretation)
    findings.append(interpretation)
else:
    print('Line plot skipped: a date field and revenue/orders field are required.')

# 2. BAR CHART — city performance.
if city_col and revenue_col:
    city_revenue = df.groupby(city_col)[revenue_col].mean().sort_values(ascending=False)
    plt.figure(figsize=(11, 5))
    sns.barplot(x=city_revenue.index, y=city_revenue.values, hue=city_revenue.index, legend=False, palette='viridis')
    plt.title('Average Revenue by City')
    plt.xlabel('City')
    plt.ylabel('Average Revenue')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    interpretation = f'Bar chart: {city_revenue.index[0]} has the highest average revenue ({city_revenue.iloc[0]:,.2f}).'
    print(interpretation)
    findings.append(interpretation)

# 3. BAR CHART — cuisine performance.
if cuisine_col and revenue_col:
    cuisine_revenue = df.groupby(cuisine_col)[revenue_col].mean().sort_values(ascending=False).head(10)
    plt.figure(figsize=(11, 5))
    sns.barplot(x=cuisine_revenue.index, y=cuisine_revenue.values, hue=cuisine_revenue.index, legend=False, palette='magma')
    plt.title('Top 10 Cuisines by Average Revenue')
    plt.xlabel('Cuisine')
    plt.ylabel('Average Revenue')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    interpretation = f'Cuisine bar chart: {cuisine_revenue.index[0]} leads the displayed cuisines with average revenue of {cuisine_revenue.iloc[0]:,.2f}.'
    print(interpretation)
    findings.append(interpretation)

# 4. SCATTER PLOT — marketing spend versus revenue.
if marketing_col and revenue_col:
    plot_data = df[[marketing_col, revenue_col]].dropna()
    plt.figure(figsize=(8, 5))
    sns.regplot(data=plot_data, x=marketing_col, y=revenue_col, scatter_kws={'alpha': 0.65}, line_kws={'color': 'red'}, label='Trend line')
    plt.title('Marketing Spend vs Revenue')
    plt.xlabel('Marketing Spend')
    plt.ylabel('Revenue')
    plt.legend()
    plt.tight_layout()
    plt.show()
    corr = plot_data[marketing_col].corr(plot_data[revenue_col])
    interpretation = f'Scatter plot: marketing spend and revenue have a {"positive" if corr >= 0 else "negative"} correlation of {corr:.2f}.'
    print(interpretation)
    findings.append(interpretation)

# 5. SCATTER PLOT — delivery time versus customer rating.
if delivery_col and rating_col:
    plot_data = df[[delivery_col, rating_col]].dropna()
    plt.figure(figsize=(8, 5))
    sns.regplot(data=plot_data, x=delivery_col, y=rating_col, scatter_kws={'alpha': 0.65, 'color': '#2a9d8f'}, line_kws={'color': '#e76f51'}, label='Trend line')
    plt.title('Delivery Time vs Customer Rating')
    plt.xlabel('Delivery Time')
    plt.ylabel('Customer Rating')
    plt.legend()
    plt.tight_layout()
    plt.show()
    corr = plot_data[delivery_col].corr(plot_data[rating_col])
    interpretation = f'Delivery-time scatter plot: delivery time and customer rating have a {"positive" if corr >= 0 else "negative"} correlation of {corr:.2f}.'
    print(interpretation)
    findings.append(interpretation)

# 6. HISTOGRAM — revenue distribution.
if revenue_col:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[revenue_col].dropna(), bins=25, kde=True, color='#457b9d')
    plt.title('Distribution of Revenue')
    plt.xlabel('Revenue')
    plt.ylabel('Number of Records')
    plt.tight_layout()
    plt.show()
    interpretation = f'Histogram: median revenue is {df[revenue_col].median():,.2f} versus a mean of {df[revenue_col].mean():,.2f}, indicating a {"right-skewed" if df[revenue_col].mean() > df[revenue_col].median() else "left-skewed or balanced"} distribution.'
    print(interpretation)
    findings.append(interpretation)

# 7. BOX PLOT — revenue differences by city.
if city_col and revenue_col:
    plt.figure(figsize=(12, 5))
    sns.boxplot(data=df, x=city_col, y=revenue_col, hue=city_col, legend=False, palette='Set2')
    plt.title('Revenue Distribution by City')
    plt.xlabel('City')
    plt.ylabel('Revenue')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    print('Box plot: compare the medians, spread, and isolated points to identify cities with more variable revenue or unusually large orders.')

# 8. VIOLIN PLOT — delivery-time distribution by cuisine.
if cuisine_col and delivery_col:
    top_cuisines = df[cuisine_col].value_counts().head(8).index
    violin_data = df[df[cuisine_col].isin(top_cuisines)]
    plt.figure(figsize=(12, 5))
    sns.violinplot(data=violin_data, x=cuisine_col, y=delivery_col, hue=cuisine_col, legend=False, palette='Pastel1', inner='quartile')
    plt.title('Delivery-Time Distribution by Cuisine (Top 8 by Order Count)')
    plt.xlabel('Cuisine')
    plt.ylabel('Delivery Time')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    slowest_cuisine = violin_data.groupby(cuisine_col)[delivery_col].mean().idxmax()
    interpretation = f'Violin plot: {slowest_cuisine} has the highest average delivery time among the displayed cuisines.'
    print(interpretation)
    findings.append(interpretation)

# 9. COUNT PLOTS — orders by channel and weather.
for column, title in [(channel_col, 'Orders by Order Channel'), (weather_col, 'Orders by Weather Condition')]:
    if column:
        order = df[column].value_counts().index
        plt.figure(figsize=(9, 5))
        sns.countplot(data=df, x=column, order=order, hue=column, legend=False, palette='Set3')
        plt.title(title)
        plt.xlabel(column)
        plt.ylabel('Number of Orders')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
        top_group = df[column].value_counts().index[0]
        interpretation = f'Count plot: {top_group} is the most frequent value for {column}.'
        print(interpretation)
        findings.append(interpretation)

# 10. CORRELATION HEATMAP — numerical business metrics.
numeric_data = df.select_dtypes(include=np.number)
if numeric_data.shape[1] >= 2:
    correlation = numeric_data.corr()
    plt.figure(figsize=(max(8, numeric_data.shape[1]), max(6, numeric_data.shape[1] * 0.75)))
    sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, linewidths=.5)
    plt.title('Correlation Heatmap of Numerical Performance Metrics')
    plt.tight_layout()
    plt.show()
    pairs = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool)).stack()
    if not pairs.empty:
        pair = pairs.abs().idxmax()
        value = pairs.loc[pair]
        interpretation = f'Correlation heatmap: the strongest numerical relationship is {pair[0]} vs {pair[1]} ({value:.2f}).'
        print(interpretation)
        findings.append(interpretation)

print('\n' + '=' * 90)
print('SUMMARY OF MOST IMPORTANT FINDINGS')
print('=' * 90)
if findings:
    for number, finding in enumerate(findings[:8], start=1):
        print(f'{number}. {finding}')
else:
    print('No requested fields were identified automatically. Check the column headers and update the candidate names near the top of the cell.')